In [ ]:
import pandas as pd
from openai import OpenAI
from autoddg import AutoDDG
from autoddg.utils import get_sample
from autoddg.evaluation import BaseEvaluator
from typing import Optional
# --- Import custom files ---
from prompts import ALL_RELATED_WORK_PROMPTS
from utils import log_result 
import os 
import json
from cache_utils import run_with_caching, load_profile_from_cache#, MockAutoDDG           #Mock is for testing

In [ ]:
# --- LLM Config ---
MODEL_CONFIG = {
    "base_url": "http://localhost:11434/v1",
    "api_key": "ollama",
    "model_name": "llama3.1:8b",
}

# --- Experiment Config ---
# --- Experiment Config ---
DATABASE_PATH_ = '../src/autoddg/database.json'
RESULTS_FILE_RELATIVE = 'autoddg_experiment_results.csv' # Keep the relative part
PROFILE_CACHE_DIR = 'profile_cache'

# Define necessary base directories
script_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
DATABASE_PATH = os.path.join(script_dir, DATABASE_PATH_)
PROJECT_ROOT = os.path.abspath(os.path.join(script_dir, os.pardir))

# --- NEW: Define the ABSOLUTE path for the results file ---
# We assume the results file path is relative to the directory where the current script is.
RESULTS_FILE_ABSOLUTE = os.path.join(script_dir, RESULTS_FILE_RELATIVE)

# Rename the variable used in your logging
RESULTS_FILE = RESULTS_FILE_ABSOLUTE
# DATABASE_PATH_ = '../src/autoddg/database.json'  # Ensure this path is correct\n",
# RESULTS_FILE = 'prompt-experiments/autoddg_experiment_results.csv'
# PROFILE_CACHE_DIR = 'profile_cache' # Directory to save/load profiles\n",

# script_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
# DATABASE_PATH = os.path.join(script_dir, DATABASE_PATH_)
# PROJECT_ROOT = os.path.abspath(os.path.join(script_dir, os.pardir))
# ABSOLUTE_CACHE_DIR = os.path.join(script_dir, PROFILE_CACHE_DIR)


# --- Define Evaluation Class ---
class Eval(BaseEvaluator):
    def __init__(self, model_name: str = MODEL_CONFIG["model_name"]):
        client = OpenAI(
            api_key=MODEL_CONFIG["api_key"], 
            base_url=MODEL_CONFIG["base_url"]
        )
        super().__init__(client=client, model_name=model_name)

# Initialize Core Tools
client = OpenAI(api_key=MODEL_CONFIG["api_key"], base_url=MODEL_CONFIG["base_url"])
auto_ddg = AutoDDG(client=client, model_name=MODEL_CONFIG["model_name"])
auto_ddg.set_evaluator(Eval())

In [ ]:
# Assuming DATABASE_PATH, run_with_caching, and auto_ddg are already defined

# 1. Load the database (adjust DATABASE_PATH if needed)
with open(DATABASE_PATH, 'r') as f:
    raw_database = json.load(f)
    # Convert keys to integers if they are dataset IDs
    database = {int(k): v for k, v in raw_database.items()}

# 2. Get the first dataset entry
try:
    # Use next(iter()) to reliably get the first key/value pair.
    # The key is the dataset_id, and the value (the info dictionary) is dataset_info.
    dataset_id, dataset_info = next(iter(database.items()))
except StopIteration:
    print("Error: The database file is empty.")
    exit()

# dataset_info is now the full metadata dictionary (e.g., {'dataset_name': '...', 'description': '...'}).

print(f"Running experiment on first dataset: ID='{dataset_id}'")

# 3. Call the run_with_caching function
run_with_caching(dataset_id, dataset_info, auto_ddg)

In [ ]:
# Assuming 'database' is the dictionary loaded from your database.json file

dataset_id = 5569235
# Use .get() for safe lookup, which returns None if the key is not found
dataset_info = database.get(dataset_id) 

if dataset_info is None:
    print(f"Error: Dataset ID '{dataset_id}' not found in the database.")
    exit()

# If the ID is found, you can now proceed to call your runner function
print(f"Running experiment on selected dataset: ID='{dataset_id}', Name='{dataset_info['dataset_name']}'")

# 3. Call the run_with_caching function
run_with_caching(dataset_id, dataset_info, auto_ddg)

In [ ]:
database = None
try:
    # Load the database JSON
    with open(DATABASE_PATH, 'r') as f:
        database = json.load(f)
        print(database)
except FileNotFoundError:
    print(f"ERROR: Database file not found at {DATABASE_PATH}")
except json.JSONDecodeError:
    print(f"ERROR: Could not decode JSON from {DATABASE_PATH}. Is the file correctly formatted?")
except Exception as e:
    print(f"An unexpected error occurred while loading the database: {e}")

# Iterate through the database entries
for dataset_id, dataset_info in database.items():
    try:
        # This calls the function that implements the cache-first logic
        run_with_caching(dataset_id, dataset_info, auto_ddg) 
    except Exception as e:
        print(f"\n{'#'*50}")
        print(f"FATAL ERROR: FAILED ON DATASET ID {dataset_id} ({dataset_info.get('dataset_name', 'Unknown')})")
        print(f"Error: {e}")
        print(f"{'#'*50}\n")
        # Continue to the next dataset instead of stopping the whole loop
        continue

print("\n=======================================================")
print("ALL CACHING RUNS FINISHED.")
print("=======================================================")


In [ ]:
test_id = '3222451'
test_profile = load_profile_from_cache(test_id)